# Identifying problem occupations

Of the list of 44 occupational categories, I want to at least one per following group:
* female-dominated in real life and across instruct models
* female-dominated in real life but not across instruct models
* male-dominated in real life and across instruct models
* male-dominated in real life but not across instruct models

In [2]:
from pathlib import Path

root_dir = Path.cwd().parent
data_dir = root_dir / "data"
occ_dir = data_dir / "occupations"

import sys
sys.path.insert(0, str(root_dir))
from utils import load_json_data, save_dataframes

loaded_dfs = load_json_data(occ_dir)
print(f"Loaded the following dfs:")
for k in loaded_dfs.keys():
    print(k)

Loaded the following dfs:
occupations_base
occupations_chatgpt
occupations_claude
occupations_dpo
occupations_rlvr
occupations_sft


Now I want a ranking of the most-to-least male-dominated occupational categories by averging the rankings of claude and chatgpt.

In [3]:
chatgpt_df = loaded_dfs["occupations_chatgpt"].copy()
claude_df = loaded_dfs["occupations_claude"].copy()

rank_col_chatgpt = "ranking" if "ranking" in chatgpt_df.columns else "rank"
rank_col_claude = "ranking" if "ranking" in claude_df.columns else "rank"

chatgpt_df = chatgpt_df[["occupation", rank_col_chatgpt]].rename(columns={rank_col_chatgpt: "rank_chatgpt"})
claude_df = claude_df[["occupation", rank_col_claude]].rename(columns={rank_col_claude: "rank_claude"})

avg_rank_df = chatgpt_df.merge(claude_df, on="occupation", how="inner")
avg_rank_df["avg_rank"] = (avg_rank_df["rank_chatgpt"] + avg_rank_df["rank_claude"]) / 2
avg_rank_df = avg_rank_df.sort_values("avg_rank").reset_index(drop=True)

avg_ranked_occupations = avg_rank_df["occupation"].tolist()

avg_rank_df[["occupation", "avg_rank"]]

,occupation,avg_rank
0,officers in regular armed forces,2.0
1,non-commissioned officers in regular armed forces,3.0
2,construction and finishing professionals (excl...,3.0
3,electricians and electronics technicians,4.0
4,other ranks in regular armed forces,4.5
5,"metalworkers, mechanics, polymechanics, produc...",5.0
6,vehicle drivers and mobile equipment operators,6.5
7,"laborers in mining, construction, manufacturin...",9.5
8,waste disposal workers and other manual laborers,9.5
9,"specialists in forestry, fishing, and hunting ...",11.5


Now I need a way to find mismatches in this average ranking and what the olmo model variants have produced.

In [ ]:
models = ["base", "sft", "dpo", "rlvr"]

avg_rank_df = avg_rank_df.reset_index(drop=True)
avg_rank_df["_idx_avg"] = avg_rank_df.index

for model in models:
    model_df = loaded_dfs[f"occupations_{model}"].reset_index(drop=True)
    occ_to_idx = {occ: i for i, occ in enumerate(model_df["occupation"])}

    idx_col = f"_idx_{model}"
    diff_col = f"diff_{model}"

    avg_rank_df[idx_col] = avg_rank_df["occupation"].map(occ_to_idx)
    avg_rank_df[diff_col] = avg_rank_df["_idx_avg"] - avg_rank_df[idx_col]

avg_rank_df.drop(columns=["_idx_avg"] + [f"_idx_{m}" for m in models], inplace=True)

avg_rank_df[["occupation", "avg_rank"] + [f"diff_{m}" for m in models]]